# Lab 06 — Credit Risk + Fairness Basics
**Responsible ML Track** · Intermediate · ~50 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Compute group-wise accuracy and TPR
2. Calculate disparate impact ratio
3. Understand why equal accuracy ≠ fair
4. Sketch a reweighing mitigation

## Datasets (this folder)
- `german_credit_prepared.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-02-classical-machine-learning/labs/lab-06-credit-fairness-preview/lab-06-credit-fairness-preview.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Paste the lab token from the ViLabs page into `VL_TOKEN` in **Cell 0**, then run that cell first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 2).
4. Do not modify the validation function. Write your code in the exercise cells, run those cells to verify, then run the validation cell. It checks the logic of your function against small cases. A different implementation is fine. It does not require the same lines of code or an exact number. If a check fails, the lab page says what is wrong and what to change. It does not give the answer.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`german_credit_prepared.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-06-credit-fairness-preview"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-02-classical-machine-learning/labs/lab-06-credit-fairness-preview/bundle/dataset.zip"
NEED = ["german_credit_prepared.csv"]  # unzipped files used by the code below


import json as _json
import urllib.parse as _urlparse
import urllib.request as _urlrequest

VLABS_API = "https://prod-api.vilabs.app"
VL_TOKEN = ""  # paste the lab token shown on the ViLabs page after Start Lab

def vlabs_submit():
    """Validation function. Do not modify."""
    import inspect
    token = str(VL_TOKEN or "").strip()
    if not token:
        print("ViLabs: paste the lab token into VL_TOKEN in Cell 0, then re-run this cell.")
        return None
    names = ["fpr_female", "fpr_male", "female_tpr"]
    answers = {}
    missing = []
    for name in names:
        fn = globals().get(name)
        if not callable(fn):
            missing.append(name)
            continue
        answers[name] = inspect.getsource(fn)
    if missing:
        print("ViLabs: write these functions before validating:", ", ".join(missing))
        return None
    base = VLABS_API.rstrip("/") + "/colab/labs/" + LAB_ID + "/session"
    ua = {"User-Agent": "Mozilla/5.0 (compatible; ViLabs-Colab/1.0)"}

    def _read(req):
        try:
            with _urlrequest.urlopen(req, timeout=90) as resp:
                return _json.loads(resp.read().decode())
        except _urlrequest.HTTPError as exc:
            print("ViLabs HTTP", exc.code, exc.read().decode(errors="replace")[:300])
            return None

    payload = _json.dumps({"token": token, "answers": answers}).encode()
    result = _read(_urlrequest.Request(
        base + "/validate/",
        data=payload,
        headers={"Content-Type": "application/json", **ua},
    ))
    if result is None:
        return None
    print("ViLabs validated", result.get("validated"))
    for row in result.get("variables") or []:
        print(row.get("name"), "pass" if row.get("ok") else "fail")
    return result

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Intermediate: When the Scorecard Plays Favorites (Fairness Preview)

> **Scenario:** The bank's legal team reviews your credit model and asks a uncomfortable question: "Does it treat male and female applicants fairly?" Overall accuracy looks fine — 74.5%. But averages can hide a lot. You dig into performance *by group* and find the model approves female applicants at a lower rate than males, even though their default rates are similar. This lab shows why "equal accuracy" is not the same as "fair", and previews how to measure and mitigate the gap.
>
> **You will learn:** how to compute group-wise accuracy and true-positive rate (TPR), what the disparate impact ratio measures and why 0.8 is a common legal benchmark, and one concrete mitigation — per-group decision thresholds.
> **Time:** ~50 minutes. **Level:** Intermediate. **Needs:** pandas + sklearn. **Env:** 🟢 Colab only.

### Mental map

| Concept | One-line idea | Where it shows up |
|---|---|---|
| Group-wise metrics | Accuracy/TPR computed separately per demographic group | female vs male TPR |
| Disparate impact ratio | Ratio of approval rates between groups | DI < 0.8 raises legal flags |
| Equal accuracy ≠ fair | Two groups can have similar accuracy but very different error types | TPR/FPR gaps |
| Per-group thresholds | Different score cutoffs for different groups | A simple mitigation lever |

---

## 1. Group-wise metrics: accuracy and TPR by sex

A single overall accuracy can hide very different experiences for different groups. The first fairness step is always to **slice the metrics by group**. Here we train the same kind of credit model as Lab 05 and compute accuracy and the true-positive rate (TPR — fraction of actual defaulters the model flags) separately for female and male applicants.

Fairness in machine learning is not an abstract philosophical debate — it is a concrete, measurable property of models that affect real people's lives. When a credit model approves 60% of male applicants but only 49% of female applicants, that gap has real consequences: women are denied loans at higher rates, which affects their ability to buy homes, start businesses, and build wealth. The question is not whether the model is "biased" in some vague sense, but whether its *outcomes* are equitable across groups.

The first step in any fairness audit is to compute **group-wise metrics**. Overall accuracy might look fine — 74.5% sounds respectable — but that number blends two very different stories. When we slice by sex, we find that female applicants have a higher TPR (the model catches more female defaulters) but a lower approval rate (fewer women are approved overall). These two facts together reveal a pattern that the overall metric hides.

The key metrics to compute per group are:
- **Accuracy**: overall correctness within the group
- **TPR (True Positive Rate)**: fraction of actual defaulters the model flags — also called recall or sensitivity
- **FPR (False Positive Rate)**: fraction of safe applicants the model falsely flags
- **Approval rate**: fraction of applicants the model approves (predicts "no default")

A fair model should have similar TPR and FPR across groups. If one group has a much lower TPR, the model is missing defaulters in that group. If one group has a much higher FPR, the model is falsely flagging safe applicants in that group. Both are forms of unfairness.

Let us compute these metrics by sex and see what the model reveals.


In [ ]:
import matplotlib
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix

df = pd.read_csv(r"german_credit_prepared.csv")
y = (df["default"] == "Default").astype(int)
X = df.drop(columns=["default"])
cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])
clf = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

res = pd.DataFrame({"sex": X_te["sex"].values, "y": y_te.values, "pred": y_pred})
print(f"overall accuracy: {accuracy_score(y_te, y_pred):.4f}\n")
for sex, g in res.groupby("sex"):
    tn, fp, fn, tp = confusion_matrix(g["y"], g["pred"]).ravel()
    acc = accuracy_score(g["y"], g["pred"])
    tpr = tp / (tp + fn)
    fpr = fp / (fp + tn)
    approval = (g["pred"] == 0).mean()
    print(f"{sex:6s}  n={len(g):3d}  acc={acc:.4f}  TPR={tpr:.4f}  FPR={fpr:.4f}  approval_rate={approval:.4f}")


**What to notice:**
1. Overall accuracy (≈0.745) looks respectable — but it blends two different stories.
2. Female TPR (≈0.85) is actually *higher* than male (≈0.775): the model catches a larger share of female defaulters.
3. Yet the female *approval rate* (≈0.49) is lower than male (≈0.60): fewer women are approved overall. That gap is the fairness problem.

> **Pitfall:** `sex` is used here only to *audit* the model, not as a model input. Feeding protected attributes into the model is both illegal in many jurisdictions and defeats the purpose of measuring their effect.

---

## 2. The disparate impact ratio

The **disparate impact (DI) ratio** compares approval rates between groups: DI = (approval rate of the disadvantaged group) / (approval rate of the advantaged group). A DI of 1.0 means perfect parity. In US employment law, DI < 0.8 is the "four-fifths rule" threshold that raises legal scrutiny — and it is a useful benchmark anywhere.

The disparate impact ratio is one of the oldest and most widely used fairness metrics. It originated in US employment law in the 1970s, when courts needed a simple test for whether a hiring practice was discriminatory. The "four-fifths rule" states that a selection rate for any race, sex, or ethnic group that is less than four-fifths (80%) of the rate for the group with the highest selection rate is evidence of adverse impact.

The beauty of DI is its simplicity. It is a single number that captures outcome parity: are both groups being approved at similar rates? A DI of 1.0 means perfect parity. A DI of 0.82 means the disadvantaged group is approved at 82% of the rate of the advantaged group — a gap that may or may not be justified by legitimate factors.

But DI has limitations. It measures *outcome* parity, not *error* parity. A model can have DI = 1.0 and still miss defaulters in one group much more often. It also does not account for base rates: if one group genuinely has a higher default rate, a lower approval rate might be justified. For a complete fairness picture, you need to look at DI alongside TPR and FPR gaps.

In this dataset, the DI is approximately 0.82 — just below the 0.8 threshold. This means the model would draw legal scrutiny in a US employment context. The question is whether the gap is explained by legitimate factors (income, credit history) or by bias in the model or the training data.

Let us compute the DI and see how close we are to the threshold.


In [ ]:
import matplotlib
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

df = pd.read_csv(r"german_credit_prepared.csv")
y = (df["default"] == "Default").astype(int)
X = df.drop(columns=["default"])
cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])
clf = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

res = pd.DataFrame({"sex": X_te["sex"].values, "y": y_te.values, "pred": y_pred})
f = res[res["sex"] == "female"]
m = res[res["sex"] == "male"]
di = (f["pred"] == 0).mean() / (m["pred"] == 0).mean()
print(f"female approval rate: {(f['pred'] == 0).mean():.4f}")
print(f"male approval rate:   {(m['pred'] == 0).mean():.4f}")
print(f"disparate impact ratio (female/male): {di:.4f}")
print(f"four-fifths rule (>= 0.8)? {'PASS' if di >= 0.8 else 'FAIL'}")

# base rates for context
print("\nactual default rate by sex (test):")
print(res.groupby("sex")["y"].mean().round(4))


**What to notice:**
1. DI ≈ 0.82 — just under the 0.8 four-fifths threshold, so this model would draw legal scrutiny.
2. The gap is not explained by base rates: female default rate (≈0.33) is only slightly higher than male (≈0.29), yet the approval gap is much larger.
3. DI is a *ratio of rates*, so it is comparable across datasets with different overall approval levels.

> **Pitfall:** DI measures *outcome* parity, not *error* parity. A model can have DI = 1.0 and still miss defaulters in one group much more often. Always look at TPR/FPR gaps alongside DI.

---

## 3. Why equal accuracy ≠ fair, and one mitigation

Here is the trap: overall accuracy is nearly identical across groups (≈0.72 female vs ≈0.76 male), so a naive audit says "fair". But the *errors* are distributed unevenly — women are approved less often. Accuracy is a single number that averages over both classes; fairness lives in the details.

This is one of the most important lessons in fairness-aware machine learning: **equal accuracy does not imply fairness**. Accuracy is a single number that blends performance on both classes. A model can have the same accuracy for two groups while making very different *types* of errors. Women might be approved less often (lower approval rate) even though the model catches a higher fraction of female defaulters (higher TPR). These two facts are not contradictory — they are two sides of the same coin.

The reason is base rates. If women in this dataset have a slightly higher default rate, the model will (correctly) flag more women as risky. But this means fewer women are approved, even if the model is equally accurate for both groups. The approval gap is a consequence of the base-rate difference, not necessarily of model bias.

So what can we do? One concrete mitigation is **per-group thresholds**: use a different score cutoff for each group so that approval rates (or TPRs) match. This is simple, transparent, and easy to explain to regulators. The idea is to raise the threshold for the group that is over-approved and lower it for the group that is under-approved, until the outcomes are balanced.

The trade-off is that per-group thresholds can reduce TPR for the group whose threshold is raised. In this case, raising the female threshold to 0.6 brings DI from 0.82 to 0.96 — near parity — but drops female TPR from 0.85 to 0.75. There is no free lunch: the "right" threshold depends on whether the business prioritises approval parity, TPR parity, or accuracy.

Let us sweep through different female thresholds and see how DI and accuracy change.


In [ ]:
import matplotlib
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

df = pd.read_csv(r"german_credit_prepared.csv")
y = (df["default"] == "Default").astype(int)
X = df.drop(columns=["default"])
cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])
clf = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
clf.fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]
sex_te = X_te["sex"].values

print("per-group threshold sweep (raise female threshold to equalize approval):")
for t_f in [0.5, 0.55, 0.6, 0.65]:
    yp = np.where(sex_te == "female", proba >= t_f, proba >= 0.5).astype(int)
    res = pd.DataFrame({"sex": sex_te, "y": y_te.values, "pred": yp})
    f = res[res["sex"] == "female"]
    m = res[res["sex"] == "male"]
    di = (f["pred"] == 0).mean() / (m["pred"] == 0).mean()
    acc = (res["pred"] == res["y"]).mean()
    print(f"  t_female={t_f}: DI={di:.4f}  accuracy={acc:.4f}  "
          f"appr_f={(f['pred'] == 0).mean():.4f}  appr_m={(m['pred'] == 0).mean():.4f}")


**What to notice:**
1. Raising the female threshold to 0.6 lifts DI from ≈0.82 to ≈0.96 — near parity — while overall accuracy stays ≈0.75.
2. The cost is a lower female TPR (fewer female defaulters caught): mitigation trades one fairness goal against another.
3. There is no free lunch — the "right" threshold depends on whether the business prioritizes approval parity, TPR parity, or accuracy.

> **Pitfall:** Per-group thresholds are a *post-hoc* fix. The deeper question is why the model scores women lower in the first place — often the training data itself reflects historical bias. Fixing the data beats patching the threshold.

---

## Exercises

Do not modify the validation function. Write your code in the exercise cells, run those cells to verify, then run the validation cell. It checks the logic of your function against small cases. A different implementation is fine. It does not require the same lines of code or an exact number. If a check fails, the lab page says what is wrong and what to change. It does not give the answer.

### Exercise 1 — TPR by sex
Train the balanced logistic regression from section 1 and compute TPR separately for female and male applicants on the test set. Which group has the higher TPR?


**Follow-up:** What are the false-positive rates?

<details>
<summary>Hint</summary>

`confusion_matrix(g["y"], g["pred"]).ravel()` returns `(tn, fp, fn, tp)`; TPR = tp / (tp + fn).

</details>

### Exercise 2 — Disparate impact ratio
Compute the DI ratio (female approval rate / male approval rate) on the test set. Does it pass the four-fifths rule (DI ≥ 0.8)?


**Follow-up:** What are the actual default rates by sex?

<details>
<summary>Hint</summary>

DI = `(f["pred"] == 0).mean() / (m["pred"] == 0).mean()` where `f` and `m` are the per-group test frames.

</details>

### Exercise 3 — One mitigation idea
Using the per-group threshold sweep from section 3, find a female threshold that brings DI close to 1.0 while keeping overall accuracy near its original value. Report the threshold, DI, and accuracy.


**Follow-up:** Use a threshold of 0.6. Among female applicants who are actually positive, what fraction does the model catch? Return that true-positive rate, not the false-positive rate.

<details>
<summary>Hint</summary>

`np.where(sex_te == "female", proba >= t_f, proba >= 0.5)` applies different thresholds per group.

</details>

---


In [ ]:
def fpr_female():
    """Among female applicants who are actually negative, what fraction does the model predict as positive? Return that female false-positive rate, not the male rate."""
    pass


In [ ]:
def fpr_male():
    """Among male applicants who are actually negative, what fraction does the model predict as positive? Return that male false-positive rate, not the female rate."""
    pass


In [ ]:
def female_tpr():
    """Use a threshold of 0.6. Among female applicants who are actually positive, what fraction does the model catch? Return that true-positive rate, not the false-positive rate."""
    pass


In [ ]:
vlabs_submit()


### What to learn next
- Other fairness metrics: equalized odds, demographic parity, calibration by group.
- Pre-processing mitigations: reweighting or resampling the training data to remove bias at the source.
- The full fairness toolkit: `fairlearn` and `aif360` for production-grade audits.

*Files in this folder: german_credit_prepared.csv (1000 rows: default + 21 features including sex, age, credit_amount, duration_in_month).*


---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
